<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Data Visualization**


Estimated time needed: **45** minutes


In this lab, you will focus on data visualization. The dataset will be provided through an RDBMS, and you will need to use SQL queries to extract the required data.


## Objectives


After completing this lab, you will be able to:


-   Visualize the distribution of data.

-   Visualize the relationship between two features.

-   Visualize composition and comparison of data.




## Demo: How to work with database


Download the database file.


In [ ]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv

**Install and Import Necessary Python Libraries**

Ensure that you have the required libraries installed to work with SQLite and Pandas:


In [ ]:
!pip install pandas 
!pip install matplotlib

import pandas as pd
import matplotlib.pyplot as plt

**Read the CSV File into a Pandas DataFrame**

Load the Stack Overflow survey data into a Pandas DataFrame:


In [ ]:
# Read the CSV file
df = pd.read_csv('survey-data.csv')

# Display the first few rows of the data
df.head()


**Create a SQLite Database and Insert the Data**

Now, let's create a new SQLite database (`survey-data.sqlite`) and insert the data from the DataFrame into a table using the sqlite3 library:


In [ ]:
import sqlite3

# Create a connection to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Write the dataframe to the SQLite database
df.to_sql('main', conn, if_exists='replace', index=False)


# Close the connection
conn.close()


**Verify the Data in the SQLite Database**
Verify that the data has been correctly inserted into the SQLite database by running a simple query:


In [ ]:
# Reconnect to the SQLite database
conn = sqlite3.connect('survey-data.sqlite')

# Run a simple query to check the data
QUERY = "SELECT * FROM main LIMIT 5"
df_check = pd.read_sql_query(QUERY, conn)

# Display the results
print(df_check)


## Demo: Running an SQL Query


Count the number of rows in the table named 'main'


In [ ]:
QUERY = """
SELECT COUNT(*) 
FROM main
"""
df = pd.read_sql_query(QUERY, conn)
df.head()


## Demo: Listing All Tables


To view the names of all tables in the database:


In [ ]:
QUERY = """
SELECT name as Table_Name FROM sqlite_master 
WHERE type = 'table'
"""
pd.read_sql_query(QUERY, conn)


## Demo: Running a Group By Query
    
For example, you can group data by a specific column, like Age, to get the count of respondents in each age group:


In [ ]:
QUERY = """
SELECT Age, COUNT(*) as count
FROM main
GROUP BY Age
ORDER BY Age
"""
pd.read_sql_query(QUERY, conn)


## Demo: Describing a table

Use this query to get the schema of a specific table, main in this case:


In [ ]:
table_name = 'main'

QUERY = """
SELECT sql FROM sqlite_master 
WHERE name= '{}'
""".format(table_name)

df = pd.read_sql_query(QUERY, conn)
print(df.iat[0,0])


## Hands-on Lab


### Visualizing the Distribution of Data

**Histograms**

Plot a histogram of CompTotal (Total Compensation).


In [ ]:
# ---- Helpers used throughout this lab ----
import numpy as np

# Age is stored as text ranges, so map each range to an approximate number (its midpoint)
AGE_MAP = {
    'Under 18 years old': 17, '18-24 years old': 21, '25-34 years old': 29.5,
    '35-44 years old': 39.5, '45-54 years old': 49.5, '55-64 years old': 59.5,
    '65 years or older': 70,
}
AGE_ORDER = list(AGE_MAP.keys())

# TimeSearching / TimeAnswering are text ranges -> approximate minutes per day
TIME_MAP = {
    'Less than 15 minutes a day': 7.5, '15-30 minutes a day': 22.5,
    '30-60 minutes a day': 45, '60-120 minutes a day': 90, 'Over 120 minutes a day': 150,
}

def years_to_num(series):
    """YearsCodePro is text: turn 'Less than 1 year' / 'More than 50 years' into numbers."""
    return pd.to_numeric(series.replace({'Less than 1 year': '0', 'More than 50 years': '51'}),
                         errors='coerce')

def split_counts(series, top_n=None):
    """Count items in a ';'-separated multi-answer column."""
    counts = series.dropna().str.split(';').explode().str.strip().value_counts()
    return counts.head(top_n) if top_n else counts

# ---- Histogram of CompTotal ----
QUERY = "SELECT CompTotal FROM main WHERE CompTotal IS NOT NULL"
comp = pd.read_sql_query(QUERY, conn)
comp['CompTotal'] = pd.to_numeric(comp['CompTotal'], errors='coerce')

# A handful of huge values squash the chart, so plot up to the 99th percentile
limit = comp['CompTotal'].quantile(0.99)
plt.figure(figsize=(10, 5))
plt.hist(comp.loc[comp['CompTotal'] <= limit, 'CompTotal'], bins=50, color='steelblue', edgecolor='black')
plt.title('Distribution of Total Compensation (CompTotal, up to 99th percentile)')
plt.xlabel('CompTotal')
plt.ylabel('Number of respondents')
plt.show()

**Box Plots**

Plot a box plot of Age.


In [ ]:
# ---- Box plot of Age ----
QUERY = "SELECT Age FROM main WHERE Age IS NOT NULL"
age = pd.read_sql_query(QUERY, conn)
age['Age_numeric'] = age['Age'].map(AGE_MAP)

plt.figure(figsize=(6, 6))
plt.boxplot(age['Age_numeric'].dropna())
plt.title('Box Plot of Age')
plt.ylabel('Age (approximate, years)')
plt.xticks([1], ['Respondents'])
plt.show()

### Visualizing Relationships in Data

**Scatter Plots**

Create a scatter plot of Age and WorkExp.


In [ ]:
# ---- Scatter plot of Age vs WorkExp ----
QUERY = "SELECT Age, WorkExp FROM main WHERE Age IS NOT NULL AND WorkExp IS NOT NULL"
aw = pd.read_sql_query(QUERY, conn)
aw['Age_numeric'] = aw['Age'].map(AGE_MAP)
aw['WorkExp'] = pd.to_numeric(aw['WorkExp'], errors='coerce')
aw = aw.dropna()

# Age has only a few values, so add a little random jitter to stop the points sitting on top of each other
jitter = np.random.uniform(-2, 2, len(aw))
plt.figure(figsize=(10, 6))
plt.scatter(aw['Age_numeric'] + jitter, aw['WorkExp'], alpha=0.2)
plt.title('Age vs Work Experience')
plt.xlabel('Age (approximate)')
plt.ylabel('Work experience (years)')
plt.show()

**Bubble Plots**

Create a bubble plot of `TimeSearching` and `Frustration` using the Age column as the bubble size.


In [ ]:
# ---- Bubble plot: TimeSearching vs Frustration, bubble size = Age ----
QUERY = """
SELECT TimeSearching, Frustration, Age
FROM main
WHERE TimeSearching IS NOT NULL AND Frustration IS NOT NULL AND Age IS NOT NULL
"""
tf = pd.read_sql_query(QUERY, conn)
tf['Time_minutes'] = tf['TimeSearching'].map(TIME_MAP)
tf['Age_numeric'] = tf['Age'].map(AGE_MAP)

# Frustration is multi-select: one row per frustration
tf = tf.assign(Frustration=tf['Frustration'].str.split(';')).explode('Frustration').reset_index(drop=True)
top_frust = tf['Frustration'].value_counts().head(8).index
tf = tf[tf['Frustration'].isin(top_frust)]

bubbles = tf.groupby(['Time_minutes', 'Frustration']).agg(
    Respondents=('Age', 'size'), Mean_Age=('Age_numeric', 'mean')).reset_index()

plt.figure(figsize=(12, 7))
plt.scatter(bubbles['Time_minutes'], bubbles['Frustration'],
            s=bubbles['Mean_Age'] * 8, c=bubbles['Respondents'], cmap='viridis', alpha=0.7, edgecolors='black')
plt.colorbar(label='Number of respondents')
plt.title('Time Spent Searching vs Frustration (bubble size = average age)')
plt.xlabel('Time spent searching (approx. minutes per day)')
plt.ylabel('Frustration')
plt.tight_layout()
plt.show()

### Visualizing Composition of Data

**Pie Charts**

Create a pie chart of the top 5 databases(`DatabaseWantToWorkWith`) that respondents wish to learn next year.


In [ ]:
# ---- Pie chart of the top 5 databases respondents want to work with ----
QUERY = "SELECT DatabaseWantToWorkWith FROM main WHERE DatabaseWantToWorkWith IS NOT NULL"
db = pd.read_sql_query(QUERY, conn)
top5_db = split_counts(db['DatabaseWantToWorkWith'], 5)

plt.figure(figsize=(8, 8))
plt.pie(top5_db, labels=top5_db.index, autopct='%1.1f%%', startangle=90)
plt.title('Top 5 Databases Respondents Want to Work With')
plt.show()

**Stacked Charts** 

Create a stacked bar chart of median `TimeSearching` and `TimeAnswering` for the age group 30 to 35.


In [ ]:
# ---- Stacked bar: median TimeSearching and TimeAnswering for ages 30-35 ----
# Age is stored in ranges; 30-35 falls in the '25-34 years old' group, the closest available match
QUERY = """
SELECT TimeSearching, TimeAnswering
FROM main
WHERE Age = '25-34 years old'
"""
ta = pd.read_sql_query(QUERY, conn)
median_search = ta['TimeSearching'].map(TIME_MAP).median()
median_answer = ta['TimeAnswering'].map(TIME_MAP).median()
print("Median minutes searching:", median_search, "| answering:", median_answer)

plt.figure(figsize=(6, 6))
plt.bar(['Age 25-34'], [median_search], label='TimeSearching', color='steelblue')
plt.bar(['Age 25-34'], [median_answer], bottom=[median_search], label='TimeAnswering', color='orange')
plt.title('Median Time Searching and Answering (ages 25-34)')
plt.ylabel('Approx. minutes per day')
plt.legend()
plt.show()

### Visualizing Comparison of Data

**Line Chart**

Plot the median `CompTotal` for all ages from 45 to 60.


In [ ]:
# ---- Line chart: median CompTotal for ages 45 to 60 ----
# The age ranges covering 45-60 are '45-54 years old' and '55-64 years old'
QUERY = """
SELECT Age, CompTotal
FROM main
WHERE Age IN ('45-54 years old', '55-64 years old') AND CompTotal IS NOT NULL
"""
ac = pd.read_sql_query(QUERY, conn)
ac['CompTotal'] = pd.to_numeric(ac['CompTotal'], errors='coerce')
median_comp = ac.groupby('Age')['CompTotal'].median()
print(median_comp)

plt.figure(figsize=(8, 5))
plt.plot(median_comp.index, median_comp.values, marker='o')
plt.title('Median CompTotal for Ages 45 to 60')
plt.xlabel('Age group')
plt.ylabel('Median CompTotal')
plt.grid(True)
plt.show()

**Bar Chart**

Create a horizontal bar chart using the `MainBranch` column.


In [ ]:
# ---- Horizontal bar chart of MainBranch ----
QUERY = """
SELECT MainBranch, COUNT(*) AS count
FROM main
WHERE MainBranch IS NOT NULL
GROUP BY MainBranch
ORDER BY count
"""
mb = pd.read_sql_query(QUERY, conn)

plt.figure(figsize=(10, 5))
plt.barh(mb['MainBranch'], mb['count'], color='steelblue')
plt.title('Respondents by Main Branch')
plt.xlabel('Number of respondents')
plt.tight_layout()
plt.show()

### Summary


In this lab, you focused on extracting and visualizing data from an RDBMS using SQL queries and SQLite. You applied various visualization techniques, including:

- Histograms to display the distribution of CompTotal.
- Box plots to show the spread of ages.
- Scatter plots and bubble plots to explore relationships between variables like Age, WorkExp, `TimeSearching` and `TimeAnswering`.
- Pie charts and stacked charts to visualize the composition of data.
- Line charts and bar charts to compare data across categories.


### Close the Database Connection

Once the lab is complete, ensure to close the database connection:


In [ ]:
conn.close()

## Authors:
Ayushi Jain


### Other Contributors:
- Rav Ahuja
- Lakshmi Holla
- Malika


Copyright © IBM Corporation. All rights reserved.
